# Download and aggregate filtered activation files

Download a user-selected completions file from GCS, filter its records by template metadata, download their activation caches, and aggregate MLP, attention, and residual activations over cached token positions. GCP project and bucket settings are read from `.env`; missing Application Default Credentials trigger `gcloud auth application-default login`.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from temporal_manifolds.activations.extract_activations import load_selected_node_groups
from temporal_manifolds.utils.activation_aggregation import (
    aggregate_activation_file,
    nodes_for_classes,
)
from temporal_manifolds.utils.completion_filters import (
    download_activation_files,
    find_activation_paths,
)

## Configuration

The completions file is downloaded from the hardcoded GCS object to `COMPLETIONS_PATH` using authenticated Google Cloud APIs. Set either metadata filter to `None` to disable it. `NODE_CLASSES` unions the requested selected-node classes. Residual streams are included atomically as complete `layer_out/<layer>` tensors: `None` includes every layer, a set includes only those layers, and `set()` excludes all residual streams. `MAX_FILES` defaults to 10 to avoid an unexpectedly large download.

In [ ]:
PROMPT_FRAMING: str | None = "task_available_time"
OUTPUT_FORMAT: str | None = "strategy_steps"
COMPLETIONS_PATH = repo_root / "data" / "completions_256.jsonl"
COMPLETIONS_GCS_BUCKET = "temporal-research-bucket"
COMPLETIONS_GCS_PREFIX = "completions"
ACTIVATIONS_DIR = repo_root / "results" / "feature_geometry_after_assistant_residual_stream"
SELECTED_NODES_PATH = repo_root / "data" / "selected_nodes" / "final_500_eap_ig.pkl"
NODES_GCS_PREFIX: str | None = "eap-ig"
NODE_CLASSES: set[str] | None = None
RESIDUAL_STREAM_LAYERS: set[int] | None = None
AGGREGATION_POLICY = "all"  # assistant or all
MAX_FILES: int | None = 10
OVERWRITE = False

## Download completions

Download `gs://temporal-research-bucket/completions/completions_256.jsonl` through the authenticated Google Cloud Storage API before filtering it. An existing local file is skipped unless `OVERWRITE` is `True`.

In [ ]:
download_activation_files(
    [COMPLETIONS_PATH],
    gcs_prefix=COMPLETIONS_GCS_PREFIX,
    overwrite=OVERWRITE,
    upload_root=COMPLETIONS_PATH.parent,
    bucket_name=COMPLETIONS_GCS_BUCKET,
)
print(f"Completions file: {COMPLETIONS_PATH}")

In [ ]:
matching_paths = find_activation_paths(
    prompt_framing=PROMPT_FRAMING,
    output_format=OUTPUT_FORMAT,
    completions_path=COMPLETIONS_PATH,
    activations_dir=ACTIVATIONS_DIR,
)
paths_to_download = matching_paths if MAX_FILES is None else matching_paths[:MAX_FILES]

print(f"Found {len(matching_paths):,} matching activation files.")
print(f"Selected {len(paths_to_download):,} files for download.")
for path in paths_to_download[:10]:
    print(path)

## Download activation caches

Existing local files are skipped unless `OVERWRITE` is `True`. Activation files are downloaded from the hardcoded `conversational_after_assistant_residual_stream/results/feature_geometry_after_assistant_residual_stream` GCS path. The selected-node definitions use the repository-relative GCS convention.

In [ ]:
downloaded_paths = download_activation_files(
    paths_to_download,
    gcs_prefix=(
        "conversational_after_assistant_residual_stream/"
        "results/feature_geometry_after_assistant_residual_stream"
    ),
    overwrite=OVERWRITE,
    upload_root=ACTIVATIONS_DIR,
    bucket_name="temporal-research-bucket",
)
download_activation_files([SELECTED_NODES_PATH], gcs_prefix=NODES_GCS_PREFIX)

print(f"Download complete: {len(downloaded_paths):,} activation paths.")
print(f"Selected-node definitions: {SELECTED_NODES_PATH}")

## Aggregate cached positions

Cached tensors begin as `batch x cached positions x features`; selected attention tensors additionally retain their head-feature dimension. The same policy is applied to every included MLP, attention, and complete residual-stream tensor:

- `assistant`: keep the first cached position.
- `all`: average every cached token position.

In [ ]:
if not downloaded_paths:
    raise ValueError("No activation files were selected.")

selected_node_groups = load_selected_node_groups(SELECTED_NODES_PATH)
print("Available node classes:", sorted(selected_node_groups))
allowed_nodes = nodes_for_classes(selected_node_groups, NODE_CLASSES)
print("Node-class filter:", "all" if NODE_CLASSES is None else sorted(NODE_CLASSES))
print(
    "Residual-stream layers:",
    "all" if RESIDUAL_STREAM_LAYERS is None else sorted(RESIDUAL_STREAM_LAYERS),
)

aggregated_by_file = []
for path in downloaded_paths:
    aggregated_by_file.append(
        aggregate_activation_file(
            path,
            AGGREGATION_POLICY,
            allowed_nodes,
            RESIDUAL_STREAM_LAYERS,
        )
    )

print(f"Aggregated {len(aggregated_by_file):,} files with policy={AGGREGATION_POLICY!r}.")
for activation_type, tensors in aggregated_by_file[0]["activations"].items():
    example_shapes = {name: tuple(tensor.shape) for name, tensor in list(tensors.items())[:3]}
    print(activation_type, example_shapes)
print("Retained selected nodes:", {
    name: len(indices)
    for name, indices in aggregated_by_file[0]["node_indices"].items()
})
print("Included residual streams:", aggregated_by_file[0]["included_residual_streams"])